In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import soundfile as sf

from datasets import Dataset
from transformers import AutoProcessor

# -----------------------------
# Paths
# -----------------------------
HOME = Path.home()

REC_ROOT   = HOME / "data_team"
AUDIO_DIR  = REC_ROOT / "audio"
REFS_CSV   = REC_ROOT / "refs.csv"

OUT_ROOT = HOME / "data_recorded_hf"
OUT_DS   = OUT_ROOT / "hf_dataset_recorded_whisper"   # train/ eval 저장될 루트
OUT_DS.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Model / Processor (Zeroth와 동일)
# -----------------------------
MODEL_ID = "openai/whisper-small"
processor = AutoProcessor.from_pretrained(MODEL_ID)

# -----------------------------
# CSV -> DataFrame
# -----------------------------
df = pd.read_csv(REFS_CSV)

# Zeroth와 동일하게 컬럼명 통일
# expected: file, transcript
if "file" not in df.columns or "transcript" not in df.columns:
    raise ValueError(f"refs.csv must have columns: file, transcript (got {list(df.columns)})")

df = df.copy()
df["file"] = df["file"].astype(str).str.strip()
df["transcript"] = df["transcript"].astype(str).fillna("").str.strip()

# 빈 값 제거
df = df[(df["file"].str.len() > 0) & (df["transcript"].str.len() > 0)].reset_index(drop=True)

# 실제 wav 경로 생성 (Zeroth의 wav_path 패턴과 동일)
df["wav_path"] = df["file"].apply(lambda x: (AUDIO_DIR / x).resolve())

# 존재하는 파일만
df = df[df["wav_path"].apply(lambda p: p.exists())].reset_index(drop=True)

if len(df) < 2:
    raise ValueError("Not enough recorded samples after filtering. Add more wav/transcripts.")

print("usable rows:", len(df))
print(df.head())

# -----------------------------
# train/eval split (Zeroth의 train/test 분리와 동일 개념)
# -----------------------------
from sklearn.model_selection import train_test_split
train_df, eval_df = train_test_split(df, test_size=0.1, random_state=42, shuffle=True)

# -----------------------------
# HF Dataset 만들기 (Zeroth의 to_hf_dataset 그대로)
# -----------------------------
def to_hf_dataset(_df: pd.DataFrame):
    return Dataset.from_dict({
        "audio_path": [str(p) for p in _df["wav_path"]],
        "text": _df["transcript"].astype(str).tolist()
    })

ds_train = to_hf_dataset(train_df)
ds_eval  = to_hf_dataset(eval_df)

print(ds_train, ds_eval)

# -----------------------------
# prepare_batch (Zeroth와 동일: sf.read -> processor -> labels)
# -----------------------------
def prepare_batch(batch):
    wav_path = batch["audio_path"]
    audio_array, sr = sf.read(wav_path)

    # stereo -> mono
    if isinstance(audio_array, np.ndarray) and audio_array.ndim > 1:
        audio_array = audio_array.mean(axis=1)

    batch["input_features"] = processor(
        audio_array,
        sampling_rate=sr
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

# -----------------------------
# map -> prep dataset (Zeroth와 동일: remove_columns=원본 컬럼 전부)
# -----------------------------
ds_train_prep = ds_train.map(
    prepare_batch,
    remove_columns=ds_train.column_names,
    num_proc=8,
    desc="Preparing recorded train"
)

ds_eval_prep = ds_eval.map(
    prepare_batch,
    remove_columns=ds_eval.column_names,
    num_proc=8,
    desc="Preparing recorded eval"
)

print(ds_train_prep, ds_eval_prep)
print("train_prep columns:", ds_train_prep.column_names)
print("eval_prep  columns:", ds_eval_prep.column_names)

# -----------------------------
# Save (Zeroth와 동일)
# -----------------------------
ds_train_prep.save_to_disk(OUT_DS / "train")
ds_eval_prep.save_to_disk(OUT_DS / "eval")

print("Saved to:", OUT_DS)
print((OUT_DS / "train").exists(), (OUT_DS / "eval").exists())


usable rows: 30
                 file                                         transcript  \
0  student01/0001.wav  아, 네… 전화 주셔서 고맙고요, 오늘은 좀 피곤한데 아침 7시에 일어나서 약 2알...   
1  student01/0002.wav  음… 그냥요, 어제는 밤 11시쯤 잠들었는데 중간에 한 번 깨서 그런지 아직 몸이 ...   
2  student01/0003.wav        아, 네… 오늘 3월 2일 맞죠, 아침엔 날이 흐려서 마음이 좀 가라앉았어요.   
3  student01/0004.wav  음… 혼자 있다 보니 조용해서 외롭긴 한데, 그래도 점심은 12시에 국을 데워 먹었어요.   
4  student01/0005.wav         아, 네… 요즘은 무릎이 좀 걱정돼서 하루에 물 1리터는 꼭 마시려고 해요.   

                                            wav_path  
0  /home/data/bootcamp/user2/data_team/audio/stud...  
1  /home/data/bootcamp/user2/data_team/audio/stud...  
2  /home/data/bootcamp/user2/data_team/audio/stud...  
3  /home/data/bootcamp/user2/data_team/audio/stud...  
4  /home/data/bootcamp/user2/data_team/audio/stud...  
Dataset({
    features: ['audio_path', 'text'],
    num_rows: 27
}) Dataset({
    features: ['audio_path', 'text'],
    num_rows: 3
})


Preparing recorded train (num_proc=8):   0%|          | 0/27 [00:00<?, ? examples/s]

num_proc must be <= 3. Reducing num_proc to 3 for dataset of size 3.


Preparing recorded eval (num_proc=3):   0%|          | 0/3 [00:00<?, ? examples/s]

Dataset({
    features: ['input_features', 'labels'],
    num_rows: 27
}) Dataset({
    features: ['input_features', 'labels'],
    num_rows: 3
})
train_prep columns: ['input_features', 'labels']
eval_prep  columns: ['input_features', 'labels']


Saving the dataset (0/1 shards):   0%|          | 0/27 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3 [00:00<?, ? examples/s]

Saved to: /home/data/bootcamp/user2/data_recorded_hf/hf_dataset_recorded_whisper
True True
